In [ ]:
from datasets import load_dataset
from transformers import MT5Tokenizer, MT5ForConditionalGeneration
from tqdm import tqdm
import pandas as pd
import torch
import os

# Kiểm tra GPU
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cuda" )
print(f"💻 Đang sử dụng thiết bị: {device}")

# Load model paraphrase và đưa lên GPU
CKPT = 'chieunq/vietnamese-sentence-paraphase'
tokenizer = MT5Tokenizer.from_pretrained(CKPT)
model = MT5ForConditionalGeneration.from_pretrained(CKPT).to(device)

# Hàm tạo paraphrase
def paraphrase(text, num_return_sequences=5):
    inputs = tokenizer(text, padding='longest', max_length=512, truncation=True, return_tensors='pt')
    inputs = {key: val.to(device) for key, val in inputs.items()}
    output = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=512,
        num_return_sequences=num_return_sequences,
        do_sample=True,
        top_p=0.95
    )
    return [tokenizer.decode(o, skip_special_tokens=True) for o in output]

# Load dataset
dataset = pd.read_parquet("../generate_dataset/SVYKHOA_dataset_diagnosis.parquet", engine="fastparquet")

# File Excel đầu ra
output_file = "SVYKHOA_dataset_diagnosis_3.xlsx"

# Tạo file Excel rỗng nếu chưa có
if not os.path.exists(output_file):
    df_empty = pd.DataFrame(columns=["STT CHƯƠNG", "MÃ CHƯƠNG", "TÊN NHÓM CHÍNH", "MÃ BỆNH", "TÊN BỆNH", "intruction", "question", "symptom", "diagnosis", "document/title", "document/description", "cme/title", "cme/description"])
    df_empty.to_excel(output_file, index=False)

# Đọc số dòng đã có để tiếp tục từ đó
existing_df = pd.read_excel(output_file)
# start_index = len(existing_df)
start_index = 5000
print(f"🚀 Bắt đầu từ dòng {start_index}")

# Số lượng mẫu muốn xử lý thêm
max_samples = 273500 - 50000  # Có thể chỉnh: 10, 100, 500...

# Duyệt dataset từ start_index
for i, row in dataset.iterrows():
    if i < start_index:
        continue
    if i >= start_index + max_samples:
        break
    print(row)

    question = row["question"]   # row là Series
    try:
        paraphrases = paraphrase(question, num_return_sequences=10)
    except Exception as e:
        print(f"Lỗi paraphrase tại mẫu {i}: {e}")
        paraphrases = [question]

    new_rows = []
    for pq in paraphrases:
        new_rows.append({
            "STT CHƯƠNG": row["STT CHƯƠNG"],
            "MÃ CHƯƠNG": row["MÃ CHƯƠNG"],
            "TÊN NHÓM CHÍNH": row["TÊN NHÓM CHÍNH"],
            "MÃ BỆNH": row["MÃ BỆNH"],
            "TÊN BỆNH": row["TÊN BỆNH"],
            "intruction": row["intruction"],
            "question": pq,
            "symptom": row["symptom"],
            "diagnosis": row["diagnosis"],
            "document/title": row["document/title"],
            "document/description": row["document/description"],
            "cme/title": row["cme/title"],
            "cme/description": row["cme/description"]
        })

    new_df = pd.DataFrame(new_rows)
    with pd.ExcelWriter(output_file, mode="a", engine="openpyxl", if_sheet_exists="overlay") as writer:
        sheet = writer.sheets["Sheet1"]
        start_row = sheet.max_row
        new_df.to_excel(writer, header=False, index=False, startrow=start_row)


print("✅ Đã lưu toàn bộ dữ liệu paraphrase vào file Excel.")


💻 Đang sử dụng thiết bị: cuda


The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'T5Tokenizer'. 
The class this function is called from is 'MT5Tokenizer'.
You are using the default legacy behaviour of the <class 'transformers.models.mt5.tokenization_mt5.MT5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


🚀 Bắt đầu từ dòng 5000
STT CHƯƠNG                                                             II
MÃ CHƯƠNG                                                         C00-D48
TÊN NHÓM CHÍNH                                                  U ác tính
MÃ BỆNH                                                             C96.3
TÊN BỆNH                                          U lympho mô bào thực sự
intruction              Chatbot y khoa chuyên về y khoa, cung cấp thôn...
question                Sau điều trị U lympho mô bào thực sự (C96.3), ...
symptom                 Bệnh nhân nam, 45 tuổi, đã hoàn thành phác đồ ...
diagnosis               Sau khi hoàn thành điều trị U lympho mô bào th...
document/title          Hướng dẫn theo dõi và quản lý tái phát U lymph...
document/description    Tài liệu này cung cấp các khuyến nghị chi tiết...
cme/title               Quản lý dài hạn và chăm sóc giảm nhẹ cho bệnh ...
cme/description         Khóa học này tập trung vào các chiến lược quản...
Name: 5000, dty